# 02 | Pandas for Sales and Profit Analysis

**Amity University, Lucknow | MBA workshop | 9 October 2026**

![Python](https://img.shields.io/badge/Python-3-3776AB?logo=python&logoColor=white)
![NumPy](https://img.shields.io/badge/NumPy-Arrays-013243?logo=numpy&logoColor=white)
![Pandas](https://img.shields.io/badge/Pandas-Analytics-150458?logo=pandas&logoColor=white)
![Audience](https://img.shields.io/badge/Audience-MBA_students-00897B)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zaid-kamil/Data-analytics-using-numpy-and-python-at-Amity-Lucknow/blob/main/notebooks/02_pandas_sales_analysis.ipynb)

**Core session: 40 minutes + 10-minute exercise.** Move from raw transactions to a management summary.

### Your instructor: Zaid Kamil

I have over 12 years of experience in software development, technical education, and technology leadership. My expertise includes Python, backend development, AI and machine learning, automation, Flutter, and Android development. I have authored two technology books published by BPB Publications: **My First Mobile App for Students** and **Flutter Solutions for Web Developers**.


## How to use this notebook
Run cells from top to bottom with **Shift + Enter**. Predict the result before running each example. All data is fictional and included in the notebook; no downloads or accounts beyond your notebook environment are needed.


## Learning outcomes
Load and inspect a table, identify quality issues, create business metrics, and compare performance by region and product.


## 1. Load a small CSV
The CSV is embedded so the example also works in Colab. `pd.read_csv("sales.csv")` follows the same pattern for a local file. Each row is a transaction; price and cost are **per unit**, in INR.


In [1]:
import pandas as pd
from io import StringIO

csv_data = """Order_ID,Date,Product,Region,Units,Unit_Price,Unit_Cost
1001,2026-09-01,Notebook,Lucknow,20,100,60
1002,2026-09-02,Pen,Delhi,50,20,10
1003,2026-09-03,Notebook,Kanpur,15,100,60
1004,2026-09-04,Planner,Lucknow,10,300,180
1005,2026-09-05,Planner,Delhi,8,300,180
1006,2026-09-06,Pen,Kanpur,40,20,10
1007,2026-09-07,Notebook,Delhi,30,100,60
1008,2026-09-08,Planner,Lucknow,,300,180
1009,2026-09-09,Pen,Lucknow,60,20,10
1010,2026-09-10,Planner,Kanpur,5,300,180
1003,2026-09-03,Notebook,Kanpur,15,100,60
"""
raw = pd.read_csv(StringIO(csv_data))
raw.head()


   Order_ID        Date   Product   Region  Units  Unit_Price  Unit_Cost
0      1001  2026-09-01  Notebook  Lucknow   20.0         100         60
1      1002  2026-09-02       Pen    Delhi   50.0          20         10
2      1003  2026-09-03  Notebook   Kanpur   15.0         100         60
3      1004  2026-09-04   Planner  Lucknow   10.0         300        180
4      1005  2026-09-05   Planner    Delhi    8.0         300        180


## 2. Inspect before calculating
A DataFrame is a labelled table; a Series is a labelled one-dimensional collection, such as one column.


In [2]:
print('Rows and columns:', raw.shape)
raw.info()
print('\nMissing values:')
print(raw.isna().sum())
print('\nRepeated order IDs:', raw['Order_ID'].duplicated().sum())
raw.describe()


Rows and columns: (11, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Order_ID    11 non-null     int64  
 1   Date        11 non-null     object 
 2   Product     11 non-null     object 
 3   Region      11 non-null     object 
 4   Units       10 non-null     float64
 5   Unit_Price  11 non-null     int64  
 6   Unit_Cost   11 non-null     int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 748.0+ bytes

Missing values:
Order_ID      0
Date          0
Product       0
Region        0
Units         1
Unit_Price    0
Unit_Cost     0
dtype: int64

Repeated order IDs: 1
          Order_ID      Units  Unit_Price   Unit_Cost
count    11.000000  10.000000   11.000000   11.000000
mean   1005.272727  25.300000  150.909091   90.000000
std       2.969542  18.968102  122.756226   74.296702
min    1001.000000   5.000000   20.000000   10.000000
2

## 3. Clean with a stated business rule
This example assumes each Order_ID identifies one transaction. Keep the first repeated ID. Quarantine the row with missing Units rather than inventing sales; confirm it with the source before including it. Real orders with multiple line items need a different deduplication key.


In [3]:
deduplicated = raw.drop_duplicates(subset=['Order_ID']).copy()
review_needed = deduplicated.loc[deduplicated['Units'].isna()].copy()
sales = deduplicated.dropna(subset=['Units']).copy()
sales['Date'] = pd.to_datetime(sales['Date'])
sales['Units'] = sales['Units'].astype(int)
print('Usable transactions:', len(sales))
print('Transactions needing review:', len(review_needed))
review_needed


Usable transactions: 9
Transactions needing review: 1
   Order_ID        Date  Product   Region  Units  Unit_Price  Unit_Cost
7      1008  2026-09-08  Planner  Lucknow    NaN         300        180


## 4. Create revenue and profit columns
Revenue = Units × Unit Price. Gross profit = Revenue − product cost. This excludes overheads, taxes, and other operating expenses.


In [4]:
sales['Revenue'] = sales['Units'] * sales['Unit_Price']
sales['Total_Cost'] = sales['Units'] * sales['Unit_Cost']
sales['Gross_Profit'] = sales['Revenue'] - sales['Total_Cost']
sales['Margin_Pct'] = (100 * sales['Gross_Profit'] / sales['Revenue'].replace(0, float('nan'))).round(2)
sales[['Order_ID', 'Product', 'Revenue', 'Gross_Profit', 'Margin_Pct']]


   Order_ID   Product  Revenue  Gross_Profit  Margin_Pct
0      1001  Notebook     2000           800        40.0
1      1002       Pen     1000           500        50.0
2      1003  Notebook     1500           600        40.0
3      1004   Planner     3000          1200        40.0
4      1005   Planner     2400           960        40.0
5      1006       Pen      800           400        50.0
6      1007  Notebook     3000          1200        40.0
8      1009       Pen     1200           600        50.0
9      1010   Planner     1500           600        40.0


## 5. Select and filter
**Example A:** view Lucknow transactions.

**Example B:** find transactions with revenue of at least INR 2,000.


In [5]:
print(sales.loc[sales['Region'] == 'Lucknow', ['Order_ID', 'Product', 'Revenue']])
sales.loc[sales['Revenue'] >= 2000, ['Order_ID', 'Region', 'Revenue']]


   Order_ID   Product  Revenue
0      1001  Notebook     2000
3      1004   Planner     3000
8      1009       Pen     1200
   Order_ID   Region  Revenue
0      1001  Lucknow     2000
3      1004  Lucknow     3000
4      1005    Delhi     2400
6      1007    Delhi     3000


## 6. Group into a management summary
Calculate regional totals and order counts. Regional margin must use total profit divided by total revenue, rather than an unweighted average of transaction percentages.


In [6]:
region_summary = sales.groupby('Region').agg(
    Revenue=('Revenue', 'sum'),
    Gross_Profit=('Gross_Profit', 'sum'),
    Orders=('Order_ID', 'count')
)
region_summary['Margin_Pct'] = (100 * region_summary['Gross_Profit'] / region_summary['Revenue']).round(2)
region_summary.sort_values('Revenue', ascending=False)


         Revenue  Gross_Profit  Orders  Margin_Pct
Region                                            
Delhi       6400          2660       3       41.56
Lucknow     6200          2600       3       41.94
Kanpur      3800          1600       3       42.11


## 7. Rank products
Ranking by revenue and ranking by profit answer different business questions.


In [7]:
product_summary = sales.groupby('Product')[['Revenue', 'Gross_Profit']].sum()
print('By revenue:')
print(product_summary.sort_values('Revenue', ascending=False))
print('\nBy profit:')
print(product_summary.sort_values('Gross_Profit', ascending=False))


By revenue:
          Revenue  Gross_Profit
Product                        
Planner      6900          2760
Notebook     6500          2600
Pen          3000          1500

By profit:
          Revenue  Gross_Profit
Product                        
Planner      6900          2760
Notebook     6500          2600
Pen          3000          1500


## 8. Optional: pivot and export
A pivot table provides an Excel-style cross-tab. The export stays in your notebook environment; in Colab, use the Files panel to download it.


In [8]:
pivot = sales.pivot_table(index='Region', columns='Product', values='Revenue', aggfunc='sum', fill_value=0)
region_summary.to_csv('regional_sales_summary.csv')
pivot


Product  Notebook   Pen  Planner
Region                          
Delhi        3000  1000     2400
Kanpur       1500   800     1500
Lucknow      2000  1200     3000


## 9. MBA mini-case
1. Which region leads in revenue?
2. Which product leads in gross profit?
3. Does the leading revenue region also lead in profit?
4. Why should the incomplete order be mentioned in your report?

Write a **two-sentence management recommendation**, separating facts from assumptions.


In [9]:
# Write your analysis here.


### Suggested answer


In [10]:
print('Highest revenue region:', region_summary['Revenue'].idxmax())
print('Highest profit product:', product_summary['Gross_Profit'].idxmax())
print('Highest profit region:', region_summary['Gross_Profit'].idxmax())
print('Revenue included:', sales['Revenue'].sum())
print('Excluded incomplete orders:', len(review_needed))


Highest revenue region: Delhi
Highest profit product: Planner
Highest profit region: Delhi
Revenue included: 16400
Excluded incomplete orders: 1


**Interpretation:** Lucknow leads revenue, while Delhi leads gross profit. Planner leads product profit. Resolve the missing Lucknow order before treating this as a final performance report. Product mix can affect profitability, so revenue alone is insufficient.

**Next:** the third notebook contains optional eigenvalue and developer-tool topics.


### Stay connected

If this session helped you, **follow [Zaid Kamil on GitHub](https://github.com/zaid-kamil)** for more learning resources and **[connect on LinkedIn](https://linkedin.com/in/zaid-kamil-94211a40)**. Mention the Amity Lucknow workshop in your connection request.

[![Follow on GitHub](https://img.shields.io/badge/GitHub-Follow_Zaid-181717?logo=github&logoColor=white)](https://github.com/zaid-kamil)
[![Connect on LinkedIn](https://img.shields.io/badge/LinkedIn-Connect-0A66C2)](https://linkedin.com/in/zaid-kamil-94211a40)

You can also star this repository to find the notebooks again.
